# Практическая работа №5: Оптимизация и очистка данных
**Вариант 4**

Задания варианта:
1. С помощью `.str.extract` извлечь `product_id` из `page` и привести его к числу.
2. Посчитать, какой процент событий связан с товарами, а какой нет.
3. Сохранить таблицу событий только с валидными `product_id` в отдельный CSV.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## Загрузка событий

In [2]:
df_events_records = pd.read_json("data/events_records.json").T.reset_index(drop=True)
df_events_records.columns = ["event_id", "user_id", "event_time", "event_type", "page"]

df_events_lines = pd.read_json("data/events_lines.json", lines=True)

df_events = pd.concat([df_events_records, df_events_lines], ignore_index=True)
df_events = df_events.astype({"event_id": "int64", "user_id": "int64", "event_type": "string", "page": "string"})
df_events["event_time"] = pd.to_datetime(df_events["event_time"])

print(df_events.head())

   event_id  user_id          event_time   event_type         page
0         1     1085 2024-01-30 20:37:32    page_view  /product/26
1         2     1006 2024-02-24 22:55:13  add_to_cart  /product/28
2         3     1024 2024-02-14 17:49:43        click  /product/27
3         4     1062 2024-02-16 11:21:32        click   /product/1
4         5     1012 2024-02-08 05:25:21  add_to_cart            /


## Задание 1. Извлечение `product_id` из `page`

In [3]:
df_events["product_id"] = df_events["page"].str.extract(r"^/product/(\d+)$", expand=False)
df_events["product_id"] = pd.to_numeric(df_events["product_id"], errors="coerce").astype("Int32")

print(df_events[["page", "product_id"]].head(10))

          page  product_id
0  /product/26          26
1  /product/28          28
2  /product/27          27
3   /product/1           1
4            /        <NA>
5  /product/19          19
6   /product/4           4
7   /product/9           9
8   /product/6           6
9  /product/12          12


## Задание 2. Доля событий, связанных и не связанных с товарами

In [4]:
is_product = df_events["product_id"].notna()

print(f"Связано с товарами: {is_product.mean() * 100:.2f}%")
print(f"Не связано с товарами: {(~is_product).mean() * 100:.2f}%")

Связано с товарами: 70.80%
Не связано с товарами: 29.20%


## Задание 3. Сохранение событий с валидными `product_id`

In [5]:
df_products = pd.read_excel("data/products.xlsx", sheet_name="products")

mask_valid = df_events["product_id"].notna() & df_events["product_id"].isin(df_products["product_id"])
df_events_valid = df_events[mask_valid]

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
df_events_valid.to_csv(results_dir / "events_valid_product_id.csv", index=False)

print(df_events_valid.shape)
print(df_events_valid.head())

(2832, 6)
   event_id  user_id          event_time   event_type         page  product_id
0         1     1085 2024-01-30 20:37:32    page_view  /product/26          26
1         2     1006 2024-02-24 22:55:13  add_to_cart  /product/28          28
2         3     1024 2024-02-14 17:49:43        click  /product/27          27
3         4     1062 2024-02-16 11:21:32        click   /product/1           1
5         6     1037 2024-02-21 19:57:26  add_to_cart  /product/19          19
